# Stage 2: Multi-Class Species Classifier

**CPTS 434 — Automated Wildlife Classification**

Direct multi-class classifier on CCT20: **15 species/categories** in one model, no two-stage pipeline.  
The training split has zero empty frames, so `empty` is excluded as a class.

**Architecture:** ResNet-50 (ImageNet pretrained), frozen backbone, `Linear(2048, 15)` head.  
**Loss:** `CrossEntropyLoss` with inverse-frequency class weights — handles severe imbalance (fox: 5 vs opossum: 2,751).  
**Primary metric:** Macro-averaged F1 (weights all 15 classes equally regardless of size).

In [ ]:
# Standard library
import os, io, json, random, copy, time
from pathlib import Path
from collections import Counter, defaultdict

# Scientific computing
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# torchvision
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.models import ResNet50_Weights

# scikit-learn metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)

# Image loading
from PIL import Image

# Progress bars
from tqdm.notebook import tqdm

print(f"PyTorch version     : {torch.__version__}")
print(f"torchvision version : {torchvision.__version__}")

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f"Seed set to {seed}")

set_seed(42)

## 1. Setup & Configuration

Same data layout as the Stage 1 notebook:

```
data/
├── eccv_18_annotation_files/
│   ├── train_annotations.json        # 13,553 images — ALL animal, 0 empty
│   ├── cis_val_annotations.json      # 3,484 images  — 28.6% empty
│   ├── cis_test_annotations.json     # 15,827 images — 7.5% empty
│   └── trans_test_annotations.json   # 23,275 images — 7.6% empty
└── eccv_18_all_images_sm/
    └── <image files>.jpg
```

Empty frames in val/test are **filtered out** before evaluation — the model was never trained on them.

In [ ]:
device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available()         else
    "cpu"
)

ANNOTATION_DIR = Path("data/eccv_18_annotation_files")
IMAGE_DIR      = Path("data/eccv_18_all_images_sm")

CONFIG = {
    "train_json"     : ANNOTATION_DIR / "train_annotations.json",
    "val_json"       : ANNOTATION_DIR / "cis_val_annotations.json",
    "cis_test_json"  : ANNOTATION_DIR / "cis_test_annotations.json",
    "trans_test_json": ANNOTATION_DIR / "trans_test_annotations.json",
    "image_dir"      : IMAGE_DIR,

    "checkpoint_dir" : Path("checkpoints/species"),
    "results_dir"    : Path("results/species"),

    "seed": 42,

    # ── Model ──────────────────────────────────────────────────────────────
    "model_name"      : "resnet50",
    "pretrained"      : True,
    "freeze_backbone" : True,
    "unfreeze_layer4" : False,   # set True to fine-tune layer4 as well

    # ── Training ───────────────────────────────────────────────────────────
    "num_epochs"          : 30,
    "batch_size"          : 64,
    "num_workers"         : 4,
    "learning_rate"       : 1e-3,
    "weight_decay"        : 1e-4,
    "lr_patience"         : 4,
    "lr_factor"           : 0.1,
    "early_stop_patience" : 8,
    "image_size"          : 224,

    "device": device,
}

set_seed(CONFIG["seed"])
CONFIG["checkpoint_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["results_dir"].mkdir(parents=True, exist_ok=True)

print(f"Device         : {CONFIG['device']}")
print(f"Checkpoint dir : {CONFIG['checkpoint_dir'].resolve()}")
print(f"Results dir    : {CONFIG['results_dir'].resolve()}")

## 2. Class Definitions

CCT20 uses **non-contiguous** category IDs (1, 3, 5 … 99).  
We build a `CAT_ID_TO_IDX` mapping to contiguous 0-based indices for `CrossEntropyLoss`.

`empty` (ID 30) is excluded — 0 training samples, so it cannot be a learned class.  
Empty frames in val/test are skipped during data loading.

In [ ]:
EMPTY_CAT_ID = 30   # excluded everywhere

# All non-empty category IDs in CCT20, in a fixed order
CLASS_IDS = [1, 3, 5, 6, 7, 8, 9, 10, 11, 16, 21, 33, 34, 51, 99]
CLASS_NAMES = [
    "opossum",   # ID  1  | 2,751 train annotations
    "raccoon",   # ID  3  | 1,120
    "squirrel",  # ID  5  | 1,256
    "bobcat",    # ID  6  |   791
    "skunk",     # ID  7  |   230
    "dog",       # ID  8  |   919
    "coyote",    # ID  9  | 1,501
    "rabbit",    # ID 10  | 2,490
    "bird",      # ID 11  |   619
    "cat",       # ID 16  | 1,290
    "badger",    # ID 21  |     9
    "car",       # ID 33  |   668
    "deer",      # ID 34  |    45
    "fox",       # ID 51  |     5
    "rodent",    # ID 99  |   377
]

CAT_ID_TO_IDX = {cat_id: idx for idx, cat_id in enumerate(CLASS_IDS)}
NUM_CLASSES   = len(CLASS_IDS)   # 15

print(f"Number of classes : {NUM_CLASSES}")
print(f"{'idx':>4}  {'cat_id':>6}  name")
print("─" * 30)
for idx, (cat_id, name) in enumerate(zip(CLASS_IDS, CLASS_NAMES)):
    print(f"{idx:>4}  {cat_id:>6}  {name}")

## 3. Data Loading

Key differences from the Stage 1 binary loader:
- Returns a **0-based class index** instead of a binary label.
- **Empty frames are skipped** (all annotations are `empty`, or no annotations at all).
- For images with **multiple annotations**, the **first non-empty annotation** determines the label.

In [ ]:
def load_cct20_split(json_path, image_dir,
                     cat_id_to_idx=CAT_ID_TO_IDX,
                     empty_cat_id=EMPTY_CAT_ID):
    """
    Load one CCT20 split for multi-class species classification.

    Parameters
    ----------
    json_path     : Path to COCO-format annotation JSON
    image_dir     : Root directory containing image files
    cat_id_to_idx : dict mapping category_id → 0-based class index
    empty_cat_id  : category ID for empty frames (skipped)

    Returns
    -------
    list of (Path, int) — image path and 0-based class index
    """
    with open(json_path) as f:
        data = json.load(f)

    image_id_to_anns = defaultdict(list)
    for ann in data["annotations"]:
        image_id_to_anns[ann["image_id"]].append(ann)

    samples       = []
    skipped_empty = 0
    skipped_unk   = 0

    for img in data["images"]:
        img_path   = Path(image_dir) / img["file_name"]
        anns       = image_id_to_anns[img["id"]]
        animal_anns = [a for a in anns if a["category_id"] != empty_cat_id]

        if not animal_anns:
            skipped_empty += 1
            continue

        cat_id = animal_anns[0]["category_id"]   # first non-empty annotation wins

        if cat_id not in cat_id_to_idx:
            skipped_unk += 1
            continue

        samples.append((img_path, cat_id_to_idx[cat_id]))

    class_counts = Counter(idx for _, idx in samples)
    max_n        = max(class_counts.values(), default=1)
    print(f"  {json_path.name}: {len(samples):,} samples "
          f"({skipped_empty:,} empty skipped)")
    for idx, name in enumerate(CLASS_NAMES):
        n   = class_counts.get(idx, 0)
        bar = "█" * int(n / max_n * 25)
        print(f"    {idx:2d}  {name:12s}: {n:5,}  {bar}")

    return samples


print("Loading CCT20 splits...")
train_samples      = load_cct20_split(CONFIG["train_json"],      CONFIG["image_dir"])
val_samples        = load_cct20_split(CONFIG["val_json"],        CONFIG["image_dir"])
cis_test_samples   = load_cct20_split(CONFIG["cis_test_json"],   CONFIG["image_dir"])
trans_test_samples = load_cct20_split(CONFIG["trans_test_json"], CONFIG["image_dir"])

print(f"\nTotals — train: {len(train_samples):,} | val: {len(val_samples):,} | "
      f"cis_test: {len(cis_test_samples):,} | trans_test: {len(trans_test_samples):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
palette   = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))

for ax, (split_name, samples) in zip(
    axes,
    [("Train", train_samples), ("Trans-Test (out-of-domain)", trans_test_samples)],
):
    counts = Counter(idx for _, idx in samples)
    values = [counts.get(i, 0) for i in range(NUM_CLASSES)]
    bars   = ax.bar(CLASS_NAMES, values, color=palette, edgecolor="white")
    for bar, v in zip(bars, values):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() * 1.05,
                    str(v), ha="center", va="bottom", fontsize=7, rotation=45)
    ax.set_title(f"{split_name}  (n={len(samples):,})", fontsize=11)
    ax.set_ylabel("Image count (log scale)")
    ax.set_yscale("log")
    ax.set_ylim(bottom=1)
    ax.spines[["top", "right"]].set_visible(False)
    plt.sca(ax)
    plt.xticks(rotation=45, ha="right", fontsize=9)

plt.suptitle("CCT20 Species Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(CONFIG["results_dir"] / "species_distribution.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved species_distribution.png")

In [ ]:
# Show up to 2 example images per species (first 8 species for brevity)
n_cols       = 4
show_classes = list(range(min(8, NUM_CLASSES)))
n_rows       = (len(show_classes) + n_cols - 1) // n_cols

class_to_paths = defaultdict(list)
for path, idx in train_samples:
    class_to_paths[idx].append(path)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
fig.suptitle("Training Set — One Example per Species", fontsize=12, fontweight="bold")

for ax_idx, ax in enumerate(axes.flat):
    if ax_idx < len(show_classes):
        cls_idx = show_classes[ax_idx]
        paths   = class_to_paths.get(cls_idx, [])
        if paths:
            try:
                ax.imshow(Image.open(paths[0]).convert("RGB"))
            except Exception:
                ax.text(0.5, 0.5, "load error", ha="center", va="center")
        ax.set_title(CLASS_NAMES[cls_idx], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(CONFIG["results_dir"] / "sample_grid.png", bbox_inches="tight", dpi=150)
plt.show()

## 4. PyTorch Dataset Class

Key change from Stage 1: labels are `torch.long` (required by `CrossEntropyLoss`),  
not `torch.float32` (which `BCEWithLogitsLoss` needed).

In [ ]:
class CameraTrapsDataset(Dataset):
    """
    Multi-class dataset returning (image_tensor, class_idx).
    Labels are torch.long for CrossEntropyLoss.
    """

    def __init__(self, samples: list, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"[WARNING] Could not load {img_path}: {e}")
            size  = CONFIG.get("image_size", 224)
            image = Image.new("RGB", (size, size), (0, 0, 0))

        if self.transform is not None:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

    def get_labels(self) -> list:
        return [label for _, label in self.samples]


print("CameraTrapsDataset defined.")

## 5. Transforms & DataLoaders

`WeightedRandomSampler` works identically for multi-class — each sample weight is  
inversely proportional to its class frequency, ensuring balanced mini-batches.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG["image_size"], scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset      = CameraTrapsDataset(train_samples,      transform=train_transform)
val_dataset        = CameraTrapsDataset(val_samples,        transform=eval_transform)
cis_test_dataset   = CameraTrapsDataset(cis_test_samples,   transform=eval_transform)
trans_test_dataset = CameraTrapsDataset(trans_test_samples, transform=eval_transform)

# ── WeightedRandomSampler ─────────────────────────────────────────────────────
train_labels   = train_dataset.get_labels()
label_counts   = Counter(train_labels)
n_total        = len(train_labels)
class_weights  = {cls: n_total / count for cls, count in label_counts.items()}
sample_weights = [class_weights[lbl] for lbl in train_labels]

sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True,
)

_pin = CONFIG["device"].type == "cuda"

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG["batch_size"],
    sampler=sampler, shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=_pin,
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=_pin,
)
cis_test_loader = DataLoader(
    cis_test_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=_pin,
)
trans_test_loader = DataLoader(
    trans_test_dataset, batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=_pin,
)

print(f"Train      : {len(train_loader):,} batches  ({len(train_dataset):,} images)")
print(f"Val        : {len(val_loader):,} batches  ({len(val_dataset):,} images)")
print(f"Cis-test   : {len(cis_test_loader):,} batches  ({len(cis_test_dataset):,} images)")
print(f"Trans-test : {len(trans_test_loader):,} batches  ({len(trans_test_dataset):,} images)")

In [ ]:
images, labels = next(iter(train_loader))
print(f"Image batch shape : {images.shape}   (expected [B, 3, 224, 224])")
print(f"Label batch shape : {labels.shape}   (expected [B])")
print(f"Label dtype       : {labels.dtype}   (expected torch.int64)")
print("\nLabel distribution in first batch:")
for idx, n in sorted(Counter(labels.numpy()).items()):
    print(f"  {CLASS_NAMES[idx]:12s} : {n}")

## 6. Model Definition

Only the FC head changes relative to Stage 1:

| Stage 1 (binary)         | Stage 2 (multi-class)             |
|--------------------------|-----------------------------------|
| `Linear(2048, 1)`        | `Linear(2048, 15)`                |
| `BCEWithLogitsLoss`      | `CrossEntropyLoss`                |
| sigmoid + threshold      | `argmax` over 15 logits           |
| recall-focused F1        | macro-averaged F1                 |

In [ ]:
def build_model(config: dict, num_classes: int) -> nn.Module:
    """
    ResNet-50 with a multi-class linear head.
    Backbone frozen by default; enable unfreeze_layer4 for deeper fine-tuning.
    """
    model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

    if config["freeze_backbone"]:
        for param in model.parameters():
            param.requires_grad = False

    if config["unfreeze_layer4"]:
        for param in model.layer4.parameters():
            param.requires_grad = True

    in_features = model.fc.in_features   # 2048
    model.fc    = nn.Linear(in_features, num_classes)
    model       = model.to(config["device"])

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Model         : ResNet-50 ({num_classes}-class head)")
    print(f"Total params  : {total_params:,}")
    print(f"Trainable     : {trainable_params:,}   (FC: {in_features} × {num_classes})")
    print(f"Frozen        : {total_params - trainable_params:,}")
    print(f"Device        : {config['device']}")

    return model


model = build_model(CONFIG, NUM_CLASSES)

## 7. Loss, Optimizer & Scheduler

**Class weights** use the standard inverse-frequency formula:

$$w_i = \frac{N}{C \cdot n_i}$$

where $N$ = total training samples, $C$ = number of classes, $n_i$ = samples in class $i$.  
This up-weights rare classes (fox: 5 → high weight) and down-weights common ones (opossum: 2,751 → low weight).

In [ ]:
def build_optimizer_and_loss(model, config, label_counts, num_classes):
    device = config["device"]
    total  = sum(label_counts.values())

    weights = []
    print("Class weights (inverse frequency):")
    for idx in range(num_classes):
        count = label_counts.get(idx, 1)   # fallback 1 avoids division by zero
        w     = total / (num_classes * count)
        weights.append(w)
        print(f"  {idx:2d}  {CLASS_NAMES[idx]:12s}: w={w:7.3f}  (n={count:,})")

    class_weight_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
    criterion           = nn.CrossEntropyLoss(weight=class_weight_tensor)

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr           = config["learning_rate"],
        weight_decay = config["weight_decay"],
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max",
        patience=config["lr_patience"],
        factor=config["lr_factor"],
    )

    print(f"\nLearning rate : {config['learning_rate']}")
    print(f"Weight decay  : {config['weight_decay']}")
    print(f"Scheduler     : ReduceLROnPlateau(mode=max, patience={config['lr_patience']})")
    return criterion, optimizer, scheduler


train_label_counts           = Counter(train_dataset.get_labels())
criterion, optimizer, scheduler = build_optimizer_and_loss(
    model, CONFIG, train_label_counts, NUM_CLASSES
)

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]", leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)              # [B, NUM_CLASSES] — raw logits
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds   = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    return running_loss / total, correct / total


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0
    all_preds    = []
    all_labels   = []

    pbar = tqdm(dataloader, desc="[Val]", leave=False)
    with torch.no_grad():
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            preds   = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return running_loss / total, correct / total, macro_f1, np.array(all_preds), np.array(all_labels)


print("train_one_epoch and validate defined.")

## 8. Training Loop

Model selection is based on **macro F1** (not accuracy).  
Top-1 accuracy on a 15-class imbalanced set is misleading — a model biased toward opossum/rabbit  
can achieve high accuracy while failing on rare species.

In [ ]:
history = {
    "train_loss": [], "val_loss"  : [],
    "train_acc" : [], "val_acc"   : [],
    "val_macro_f1": [],
}

best_val_f1              = -1.0
best_model_state         = None
epochs_without_improvement = 0
best_epoch               = 0

checkpoint_path = CONFIG["checkpoint_dir"] / "best_model.pth"

print(f"Training for up to {CONFIG['num_epochs']} epochs on {CONFIG['device']}")
print(f"Early stopping patience : {CONFIG['early_stop_patience']} epochs")
print(f"Checkpoint              : {checkpoint_path}")
print("-" * 80)

for epoch in range(1, CONFIG["num_epochs"] + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, CONFIG["device"], epoch
    )
    val_loss, val_acc, val_f1, _, _ = validate(
        model, val_loader, criterion, CONFIG["device"]
    )
    elapsed = time.time() - t0

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_f1)

    scheduler.step(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1              = val_f1
        best_epoch               = epoch
        epochs_without_improvement = 0
        best_model_state         = copy.deepcopy(model.state_dict())
        torch.save({
            "epoch"              : epoch,
            "model_state_dict"   : best_model_state,
            "optimizer_state_dict": optimizer.state_dict(),
            "val_macro_f1"       : val_f1,
            "class_names"        : CLASS_NAMES,
            "cat_id_to_idx"      : CAT_ID_TO_IDX,
            "config"             : {k: str(v) for k, v in CONFIG.items()},
        }, checkpoint_path)
        marker = "  ← best"
    else:
        epochs_without_improvement += 1
        marker = ""

    current_lr = optimizer.param_groups[0]["lr"]
    print(
        f"Epoch {epoch:3d}/{CONFIG['num_epochs']} | "
        f"Loss {train_loss:.4f}/{val_loss:.4f} | "
        f"Acc {train_acc:.3f}/{val_acc:.3f} | "
        f"Val macro-F1: {val_f1:.4f} | "
        f"LR: {current_lr:.2e} | "
        f"{elapsed:.1f}s{marker}"
    )

    if epochs_without_improvement >= CONFIG["early_stop_patience"]:
        print(f"\nEarly stopping after {epoch} epochs.")
        break

print(f"\nBest val macro-F1 = {best_val_f1:.4f} at epoch {best_epoch}")
print(f"Checkpoint saved : {checkpoint_path}")

In [ ]:
epochs_ran = list(range(1, len(history["train_loss"]) + 1))
fig, axes  = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(epochs_ran, history["train_loss"], label="Train loss", color="#4C72B0", lw=1.8)
ax.plot(epochs_ran, history["val_loss"],   label="Val loss",   color="#DD8452", lw=1.8)
ax.axvline(best_epoch, color="gray", ls="--", lw=1.2, label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Loss Curves"); ax.legend()
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
ax.plot(epochs_ran, history["train_acc"],    label="Train acc",    color="#4C72B0", lw=1.8)
ax.plot(epochs_ran, history["val_acc"],      label="Val acc",      color="#DD8452", lw=1.8)
ax.plot(epochs_ran, history["val_macro_f1"], label="Val macro-F1", color="#55A868", lw=1.8, ls="-.")
ax.axvline(best_epoch, color="gray", ls="--", lw=1.2, label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
ax.set_title("Accuracy & Macro F1"); ax.legend()
ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("Species Classifier — Training Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(CONFIG["results_dir"] / "training_curves.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved training_curves.png")

## 9. Final Evaluation on Test Sets

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=CONFIG["device"])
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} "
      f"(val macro-F1 = {checkpoint['val_macro_f1']:.4f})")

In [ ]:
for split_name, loader in [
    ("Cis-test   (in-domain)    ", cis_test_loader),
    ("Trans-test (out-of-domain)", trans_test_loader),
]:
    loss, acc, f1, preds, labels = validate(model, loader, criterion, CONFIG["device"])
    print(f"\n{'─' * 60}")
    print(f"  {split_name}")
    print(f"{'─' * 60}")
    print(f"  Loss     : {loss:.4f}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Macro F1 : {f1:.4f}")
    print()
    print(classification_report(
        labels, preds, target_names=CLASS_NAMES, digits=4, zero_division=0
    ))

# Alias trans_test as primary benchmark
test_preds  = preds
test_labels = labels

In [ ]:
cm = confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.4, ax=ax,
)
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("True",      fontsize=11)
ax.set_title("Confusion Matrix — Trans-Test (out-of-domain)", fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0,  fontsize=9)
plt.tight_layout()
cm_path = CONFIG["results_dir"] / "confusion_matrix.png"
plt.savefig(cm_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved {cm_path}")

In [ ]:
report      = classification_report(
    test_labels, test_preds, target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)
class_f1      = [report[name]["f1-score"] for name in CLASS_NAMES]
class_support = [int(report[name]["support"]) for name in CLASS_NAMES]
macro_f1_val  = report["macro avg"]["f1-score"]

fig, ax = plt.subplots(figsize=(13, 5))
colors  = ["#4C72B0" if f >= 0.5 else "#DD8452" for f in class_f1]
bars    = ax.bar(CLASS_NAMES, class_f1, color=colors, edgecolor="white", linewidth=1.1)

for bar, support in zip(bars, class_support):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f"n={support}", ha="center", va="bottom", fontsize=7, rotation=45,
    )

ax.axhline(0.5,          color="gray", ls="--", lw=1.2, label="F1 = 0.50")
ax.axhline(macro_f1_val, color="red",  ls=":",  lw=1.8,
           label=f"Macro F1 = {macro_f1_val:.3f}")
ax.set_xlabel("Species", fontsize=11)
ax.set_ylabel("F1 Score", fontsize=11)
ax.set_title("Per-Class F1 — Trans-Test (out-of-domain)", fontsize=12, fontweight="bold")
ax.set_ylim(0, 1.25)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.tight_layout()
f1_path = CONFIG["results_dir"] / "per_class_f1.png"
plt.savefig(f1_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"Macro F1 = {macro_f1_val:.4f}")
print(f"Saved {f1_path}")

## 10. Inference

In [ ]:
def predict_top_k(image_path, model, config, k: int = 3):
    """
    Return top-k predicted species with probabilities.

    Returns
    -------
    list of (species_name: str, probability: float)
    """
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(config["image_size"]),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    image  = Image.open(image_path).convert("RGB")
    tensor = transform(image).unsqueeze(0).to(config["device"])

    model.eval()
    with torch.no_grad():
        logits = model(tensor)                           # [1, NUM_CLASSES]
        probs  = torch.softmax(logits, dim=1).squeeze(0) # [NUM_CLASSES]

    topk_probs, topk_idxs = torch.topk(probs, k)
    return [(CLASS_NAMES[i.item()], p.item()) for i, p in zip(topk_idxs, topk_probs)]


print("predict_top_k() defined.")

In [ ]:
random.seed(CONFIG["seed"])
example_indices = random.sample(range(len(trans_test_samples)), 8)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Trans-Test Inference — Top-3 Predictions", fontsize=13, fontweight="bold")

for ax, idx in zip(axes.flat, example_indices):
    path, true_idx = trans_test_samples[idx]
    top3           = predict_top_k(path, model, CONFIG, k=3)
    predicted_name = top3[0][0]
    true_name      = CLASS_NAMES[true_idx]
    correct        = predicted_name == true_name

    try:
        ax.imshow(Image.open(path).convert("RGB"))
    except Exception:
        ax.text(0.5, 0.5, "load error", ha="center", va="center")

    lines = [f"True: {true_name}", "─" * 18]
    for name, prob in top3:
        lines.append(f"{name}: {prob:.1%}")
    ax.set_title("\n".join(lines), fontsize=8, color="green" if correct else "red")
    ax.axis("off")

plt.tight_layout()
plt.savefig(CONFIG["results_dir"] / "inference_examples.png", bbox_inches="tight", dpi=150)
plt.show()

## Summary

| Artifact | Path |
|----------|------|
| Best checkpoint | `checkpoints/species/best_model.pth` |
| Training curves | `results/species/training_curves.png` |
| Species distribution | `results/species/species_distribution.png` |
| Confusion matrix (trans-test) | `results/species/confusion_matrix.png` |
| Per-class F1 (trans-test) | `results/species/per_class_f1.png` |
| Inference examples | `results/species/inference_examples.png` |

### Key design decisions

- **15 classes** — all CCT20 non-empty categories, including rare ones (fox: 5, badger: 9).  
- **Inverse-frequency class weights** — heavily up-weights rare classes to prevent the model from ignoring them.  
- **WeightedRandomSampler** — ensures balanced mini-batches during training.  
- **Macro F1** — primary metric; equally penalises failures on rare and common classes.  
- **Empty frames filtered** — the training split has zero empties; val/test empties are excluded before evaluation.  
- **Trans-test** (out-of-domain cameras) is the primary benchmark — harder and more representative of real deployment.